# Resultados: 128 eletrodos × seleções de 32 (treino individual 80/20)

Lê as predições salvas por paciente (mesmo `_test.npz` para todos os modelos), aplica o Kalman e compara com Wilcoxon pareado e teste de não inferioridade (margem 0,01 de R²). Equivale ao `cache_saliency/resumo.py`.

In [ ]:
import os
import pandas as pd
from emgpred.evaluation.results import collect_results, load_patient_predictions
from emgpred.evaluation.stats import non_inferiority, paired_comparison
from emgpred.evaluation.plots import plot_paired_boxplot, plot_predictions
from emgpred.postprocessing import smooth

LOGS = "logs"     # saída da CLI; use os.environ["EMGPRED_LOGS"] para resultados de outra pasta
RUNS = {                                   # nome: (pasta, arquivo de predição)
    "128":          ("fixo_todos_8020", None),
    "32 média":     ("fixo_media_8020", None),
    "32 Sharpe":    ("fixo_sharpe_8020", None),
    "média d32":    ("fixo_media_8020_d32", None),
    "dinâmico dyn": ("autopruning_32_8020", "predictions_dyn.csv"),
    "dinâmico pat": ("autopruning_32_8020", "predictions_pat.csv"),
    "dinâmico d32": ("autopruning_32_8020_d32", "predictions_dyn.csv"),
}
KALMAN = dict(method="window", half_window=3, q=0.012, r=1.0)   # ±3 janelas
STAGE = "kalman"                                                 # raw | kalman

In [ ]:
tables = {}
for name, (folder, pred) in RUNS.items():
    path = f"{LOGS}/{folder}"
    if os.path.isdir(path):
        tables[name] = collect_results(path, kalman_kwargs=KALMAN, pred_name=pred)
r2 = pd.DataFrame({n: df.set_index("patient")[f"{STAGE}_r2"] for n, df in tables.items()})
print(r2.agg(["mean", "std", "count"]).T.round(4))
r2.round(4)

In [ ]:
pick = lambda n: tables[n][["patient", f"{STAGE}_r2", f"{STAGE}_mae"]].rename(columns=lambda c: c.replace(f"{STAGE}_", ""))
pairs = [("128", "32 média"), ("dinâmico dyn", "32 média"), ("32 média", "média d32"), ("32 média", "32 Sharpe")]
pd.concat([paired_comparison(pick(a), pick(b), labels=("A", "B")).assign(A=a, B=b)
           for a, b in pairs if a in tables and b in tables], ignore_index=True).round(4)

In [ ]:
noninf = [("média d32", "32 média"), ("32 média", "128"), ("dinâmico d32", "média d32")]
rows = []
for a, b in noninf:
    if a in tables and b in tables:
        m = pick(a).merge(pick(b), on="patient", suffixes=("_a", "_b"))
        rows.append({"A": a, "B": b, **non_inferiority(m["r2_a"], m["r2_b"], margin=0.01)})
pd.DataFrame(rows).round(4)

In [ ]:
cols = [c for c in r2.columns]
plot_paired_boxplot(r2.dropna().reset_index(), cols, ylabel=f"R² ({STAGE})");

In [ ]:
pid = "BBS"
y_true, y_pred = load_patient_predictions(f"{LOGS}/fixo_todos_8020/paciente_{pid}")
plot_predictions(y_true[:3000], smooth(y_pred, **KALMAN)[:3000], title=f"128 eletrodos — {pid}");